# Variantes de DRAEM
Cada defecto generado requiere la tapa normal original de TRAIN, la imagen alterada, la máscara alineada y una revisión humana de calidad. El objetivo reconstructivo sigue siendo la tapa original.

In [ ]:
from collections import defaultdict
import random


def synthetic_pairs(frame):
    base = DATA / "synthetic"
    pairs = pd.read_csv(base / "pairs.csv", dtype=str, keep_default_na=False)
    normal = frame[(frame.split == "train") & (frame.label == 0)].set_index("file")
    if pairs.empty or not pairs.review.eq("accepted").all():
        raise ValueError("El conjunto sintético necesita pares aprobados.")
    mapping = defaultdict(list)
    evidence_rows = []
    for pair in pairs.itertuples():
        if (
            pair.source not in normal.index
            or not pair.generator.strip()
            or not pair.defect.strip()
        ):
            raise ValueError("Revisa la procedencia, generador y tipo de defecto.")
        clean = normal.at[pair.source, "path"]
        altered, mask = [(base / value).resolve() for value in (pair.image, pair.mask)]
        if not all(p.is_relative_to(base.resolve()) for p in (altered, mask)):
            raise ValueError("Los pares deben estar dentro de data/synthetic.")
        with (
            Image.open(clean) as original,
            Image.open(altered) as generated,
            Image.open(mask) as target,
        ):
            binary = np.asarray(target.convert("L"))
            if (
                original.size != generated.size
                or original.size != target.size
                or not binary.any()
                or not set(np.unique(binary)) <= {0, 1, 255}
            ):
                raise ValueError(f"Máscara o tamaño inválido en {pair.image}.")
        mapping[str(clean)].append((altered, mask))
        evidence_rows.append(
            [pair.source, digest(altered), digest(mask), pair.generator, pair.defect]
        )
    if len({row[1] for row in evidence_rows}) != len(pairs):
        raise ValueError("Hay imágenes sintéticas repetidas.")
    return (
        mapping,
        fingerprint(evidence_rows),
        pairs.groupby(["defect", "generator"]).size(),
    )

## Aprendizaje con pares de imágenes
Todas las variantes ven una anomalía en el 50 % de los pasos, igual que DRAEM-Ref. En `draem-project` la anomalía es siempre un par generado, elegido al azar entre todos los pares aprobados, y el objetivo de reconstrucción es su tapa original. En `draem-mixed` la mitad de esas anomalías es procedural y la otra mitad son pares.

In [ ]:
def read_image(path):
    with Image.open(path) as image:
        return tv_tensors.Image(pil_to_tensor(image.convert("RGB")).float() / 255)


class PairedDraem(Draem):
    def load_pair(self, like):
        source, image_path, mask_path = random.choice(self.pairs)
        with Image.open(mask_path) as image:
            target = tv_tensors.Mask(pil_to_tensor(image.convert("L"))[0] > 0)
        original = self.pair_transform(read_image(source))
        altered, target = self.pair_transform(read_image(image_path), target)
        return (
            original.unsqueeze(0).to(like),
            altered.unsqueeze(0).to(like),
            target.unsqueeze(0).unsqueeze(0).to(like),
        )

    def training_step(self, batch, *args, **kwargs):
        clean = batch.image
        if random.random() >= 0.5:
            augmented = clean
            mask = torch.zeros_like(clean[:, :1])
        elif self.variant == "draem-mixed" and random.random() < 0.5:
            augmented, mask = self.augmenter(clean)
        else:
            clean, augmented, mask = self.load_pair(clean)
        restored, prediction = self.model(augmented)
        return self.loss(clean, restored, mask, prediction)